# Heart Disease Risk Predictor
# Feature Engineering & Data Preparation

## Objective
This notebook prepares the cleaned dataset for machine learning model development.

The preprocessing pipeline includes:
- Loading the cleaned dataset
- Separating features and target
- Train-test split
- Feature scaling
- Saving processed datasets for model training
The output of this notebook will be used directly in the machine learning modeling stage.

In [14]:
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

In [15]:
pd.set_option("display.max_columns", None)
RANDOM_STATE = 42
TEST_SIZE = 0.2
PROJECT_ROOT = Path("..")
PROCESSED_DIR = PROJECT_ROOT/"data"/"processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR = PROJECT_ROOT/"models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Load Clean Dataset
The cleaned dataset generated in Notebook 02 is loaded for feature engineering and data preparation.

In [16]:
df = pd.read_csv("../data/processed/heart_disease_clean.csv")

# Feature and Target Separation
Machine learning models require the predictor variables (features) and the target variable to be separated before training.

In [17]:
X = df.drop(columns="target")
y = df["target"]

In [18]:
print("Feature Shape :", X.shape)
print("Target Shape :", y.shape)

Feature Shape : (303, 13)
Target Shape : (303,)


In [19]:
X.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0


In [20]:
y.head()

0    0
1    1
2    1
3    0
4    0
Name: target, dtype: int64

# Feature Overview
The following features will be used as predictor variables in the machine learning models.

In [21]:
feature_summary = pd.DataFrame({
    "Feature": X.columns,
    "Data Type": X.dtypes.values
})

feature_summary

,Feature,Data Type
0,age,float64
1,sex,float64
2,cp,float64
3,trestbps,float64
4,chol,float64
5,fbs,float64
6,restecg,float64
7,thalach,float64
8,exang,float64
9,oldpeak,float64


# Target Verification
Before splitting the dataset, the target distribution is verified to ensure that both classes are represented appropriately.

In [22]:
y.value_counts()

target
0    164
1    139
Name: count, dtype: int64

In [23]:
(y.value_counts(normalize=True).mul(100).round(2))

target
0    54.13
1    45.87
Name: proportion, dtype: float64

### Observations
- Verify that both target classes are available.
- Ensure that the dataset remains balanced before train-test splitting.

# Train-Test Split
The dataset is divided into training and testing subsets.
An **80:20** split is used, while **stratification** ensures that the class distribution remains consistent across both subsets.

In [24]:
X_train, X_test, y_train, y_test = train_test_split(X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

In [25]:
print("Training Features :", X_train.shape)
print("Testing Features :", X_test.shape)
print()
print("Training Target :", y_train.shape)
print("Testing Target :", y_test.shape)

Training Features : (242, 13)
Testing Features : (61, 13)

Training Target : (242,)
Testing Target : (61,)


In [26]:
print("Training Distribution")
print(y_train.value_counts(normalize=True))
print()
print("Testing Distribution")
print(y_test.value_counts(normalize=True))

Training Distribution
target
0    0.541322
1    0.458678
Name: proportion, dtype: float64

Testing Distribution
target
0    0.540984
1    0.459016
Name: proportion, dtype: float64


### Observations
- Verify that the dataset has been successfully divided.
- Confirm that class proportions remain similar after stratification.
- The training dataset will be used for model learning, while the testing dataset will be reserved for unbiased evaluation.

# Feature Scaling
Several machine learning algorithms, such as Logistic Regression, Support Vector Machine (SVM), and K-Nearest Neighbors (KNN), are sensitive to differences in feature scales.
To ensure consistent feature magnitudes, StandardScaler is applied by fitting the scaler on the training set and transforming both the training and testing sets.

In [27]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [28]:
X_train_scaled = pd.DataFrame(X_train_scaled,
    columns=X_train.columns,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(X_test_scaled,
    columns=X_test.columns,
    index=X_test.index
)

In [29]:
X_train_scaled.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal
180,-0.729485,0.68313,0.870169,-0.395692,0.458139,-0.411196,1.022996,0.708371,-0.696177,-0.445445,0.675060,-0.689715,1.179973
208,0.050166,0.68313,-1.184278,-0.054513,0.230598,-0.411196,-0.981579,0.222495,-0.696177,-0.891627,-0.958585,-0.689715,-0.878070
167,-0.061212,-1.46385,-1.184278,0.059213,0.723605,2.431930,1.022996,0.399178,1.436416,-0.891627,-0.958585,0.445734,-0.878070
105,-0.061212,0.68313,-1.184278,-1.305501,1.121803,-0.411196,-0.981579,0.266666,-0.696177,-0.891627,-0.958585,-0.689715,1.179973
297,0.272924,-1.46385,0.870169,0.514117,-0.167601,-0.411196,-0.981579,-1.190962,1.436416,-0.713154,0.675060,-0.689715,1.179973


In [30]:
X_test_scaled.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal
219,0.495681,0.68313,0.870169,0.400391,0.401254,-0.411196,1.022996,1.415100,-0.696177,-0.891627,-0.958585,-0.689715,-0.878070
271,1.275333,0.68313,0.870169,1.651379,-0.414104,-0.411196,1.022996,-0.528404,-0.696177,1.160812,-0.958585,-0.689715,0.665462
89,-0.395349,-1.46385,-0.157055,-0.054513,0.116827,-0.411196,1.022996,-0.042528,-0.696177,-0.445445,-0.958585,-0.689715,-0.878070
101,-2.288787,0.68313,-2.211502,-0.736870,-1.286348,-0.411196,1.022996,1.061736,-0.696177,-0.891627,-0.958585,-0.689715,-0.878070
67,-0.061212,0.68313,-0.157055,1.082748,-0.338257,-0.411196,1.022996,0.664201,-0.696177,0.536156,-0.958585,-0.689715,1.179973


In [31]:
X_train_scaled.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
age,242.0,-0.0,1.0,-2.85,-0.73,0.16,0.72,2.50
sex,242.0,0.0,1.0,-1.46,-1.46,0.68,0.68,0.68
cp,242.0,0.0,1.0,-2.21,-0.93,-0.16,0.87,0.87
trestbps,242.0,-0.0,1.0,-2.10,-0.62,-0.05,0.51,3.93
chol,242.0,0.0,1.0,-2.35,-0.72,-0.10,0.53,5.96
fbs,242.0,-0.0,1.0,-0.41,-0.41,-0.41,-0.41,2.43
restecg,242.0,0.0,1.0,-0.98,-0.98,-0.98,1.02,1.02
thalach,242.0,-0.0,1.0,-3.49,-0.68,0.16,0.71,2.30
exang,242.0,0.0,1.0,-0.70,-0.70,-0.70,1.44,1.44
oldpeak,242.0,0.0,1.0,-0.89,-0.89,-0.18,0.54,4.64


### Observations
- Numerical features have been standardized.
- The transformed features are centered around zero.
- Standardized features are ready for scale-sensitive machine learning algorithms.

# Save StandardScaler
The fitted StandardScaler is saved for future inference and model deployment.
Using the same scaler ensures consistency between training data and unseen data.

In [32]:
joblib.dump(scaler, MODEL_DIR/"standard_scaler.pkl")

['..\\models\\standard_scaler.pkl']

# Save Processed Dataset
The processed datasets are exported for the machine learning modeling stage.
Both the original and standardized feature sets are preserved to support different machine learning algorithms.

In [33]:
X_train.to_csv(PROCESSED_DIR/"X_train.csv", index=False)
X_test.to_csv(PROCESSED_DIR/"X_test.csv", index=False)

In [34]:
X_train_scaled.to_csv(PROCESSED_DIR/"X_train_scaled.csv", index=False)
X_test_scaled.to_csv(PROCESSED_DIR/"X_test_scaled.csv", index=False)

In [35]:
y_train.to_csv(PROCESSED_DIR/"y_train.csv", index=False)
y_test.to_csv(PROCESSED_DIR/"y_test.csv", index=False)

# File Verification
The following files should now be available in the processed data directory.

In [36]:
sorted([file.name for file in PROCESSED_DIR.iterdir()])

['X_test.csv',
 'X_test_scaled.csv',
 'X_train.csv',
 'X_train_scaled.csv',
 'heart_disease_clean.csv',
 'y_test.csv',
 'y_train.csv']

In [37]:
loaded_scaler = joblib.load(MODEL_DIR/"standard_scaler.pkl")
loaded_scaler

,copy,True
,with_mean,True
,with_std,True


### Observations
- The preprocessing pipeline has been successfully completed.
- The StandardScaler has been saved for future use.
- The processed datasets are now ready for machine learning model development.

# Notebook Summary
This notebook completed the data preparation process required for machine learning.

The main activities included:
- Loading the cleaned dataset
- Separating features and target
- Performing train-test split
- Applying feature scaling
- Saving processed datasets
- Saving the fitted StandardScaler
The outputs generated in this notebook will be used directly in the machine learning modeling stage.

# Next Step
The next notebook focuses on building baseline machine learning models.
Multiple classification algorithms will be trained and compared to identify the best-performing model before hyperparameter tuning and explainability analysis.